# PRCP-1016: Heart Disease Prediction

## Domain: Healthcare

### Objective

The objective of this project is to perform complete data analysis on the given heart disease dataset and develop Machine Learning models to predict the presence of heart disease.

### Project Tasks

1. Prepare a complete data analysis report.
2. Create Machine Learning models for heart disease prediction.
3. Compare the performance of multiple Machine Learning models.
4. Select a suitable model for production based on the evaluation results.
5. Report the challenges faced and techniques used.
6. Provide suggestions to hospitals for early identification and prevention of potential heart disease risks.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

In [ ]:
values = pd.read_csv("Data/values.csv")
values.head()

In [ ]:
labels = pd.read_csv("Data/labels.csv")
labels.head()

In [ ]:
print("Values Dataset Shape:", values.shape)
print("Labels Dataset Shape:", labels.shape)

In [ ]:
print("Values Columns:")
print(values.columns.tolist())

print("\nLabels Columns:")
print(labels.columns.tolist())

In [ ]:
df = pd.merge(
    values,
    labels,
    on="patient_id",
    how="inner"
)

print("Merged Dataset Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
missing_percentage = df.isnull().sum() / len(df) * 100

missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": missing_percentage
})

missing_report

In [ ]:
print("Number of Duplicate Rows:", df.duplicated().sum())

In [ ]:
print("Duplicate Patient IDs:", df["patient_id"].duplicated().sum())

In [ ]:
print("Total Patients:", len(df))
print("Unique Patient IDs:", df["patient_id"].nunique())

In [ ]:
df_model = df.drop(
    "patient_id",
    axis=1
)

df_model.head()

In [ ]:
print(df_model["heart_disease_present"].value_counts())

In [ ]:
target_percentage = (
    df_model["heart_disease_present"]
    .value_counts(normalize=True)
    * 100
)

print(target_percentage)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df_model,
    x="heart_disease_present"
)

plt.title("Heart Disease Distribution")
plt.xlabel("Heart Disease Present")
plt.ylabel("Number of Patients")

plt.show()

In [ ]:
numerical_columns = df_model.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_columns.remove("heart_disease_present")

print("Numerical Columns:")
print(numerical_columns)

In [ ]:
categorical_columns = df_model.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(7, 4))

    sns.histplot(
        data=df_model,
        x=column,
        kde=True
    )

    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")

    plt.show()

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(7, 4))

    sns.boxplot(
        data=df_model,
        x=column
    )

    plt.title(f"Boxplot of {column}")
    plt.xlabel(column)

    plt.show()

In [ ]:
print(df_model["thal"].value_counts())

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(
    data=df_model,
    x="thal"
)

plt.title("Distribution of Thal")
plt.xlabel("Thal")
plt.ylabel("Number of Patients")

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(
    data=df_model,
    x="thal",
    hue="heart_disease_present"
)

plt.title("Thal vs Heart Disease")
plt.xlabel("Thal")
plt.ylabel("Number of Patients")

plt.show()

In [ ]:
categorical_like_features = [
    "sex",
    "chest_pain_type",
    "num_major_vessels",
    "fasting_blood_sugar_gt_120_mg_per_dl",
    "resting_ekg_results",
    "exercise_induced_angina",
    "slope_of_peak_exercise_st_segment"
]

for column in categorical_like_features:
    plt.figure(figsize=(7, 4))

    sns.countplot(
        data=df_model,
        x=column,
        hue="heart_disease_present"
    )

    plt.title(f"{column} vs Heart Disease")
    plt.xlabel(column)
    plt.ylabel("Number of Patients")

    plt.show()

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(7, 4))

    sns.boxplot(
        data=df_model,
        x="heart_disease_present",
        y=column
    )

    plt.title(f"{column} vs Heart Disease")

    plt.show()

In [ ]:
numeric_df = df_model.select_dtypes(
    include=["int64", "float64"]
)

correlation_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
target_correlation = (
    correlation_matrix["heart_disease_present"]
    .sort_values(ascending=False)
)

print(target_correlation)

In [ ]:
outlier_report = []

for column in numerical_columns:
    Q1 = df_model[column].quantile(0.25)
    Q3 = df_model[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    number_of_outliers = (
        (
            (df_model[column] < lower_bound) |
            (df_model[column] > upper_bound)
        ).sum()
    )

    outlier_report.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Number of Outliers": number_of_outliers
    })

outlier_df = pd.DataFrame(outlier_report)
outlier_df

# Data Analysis Findings

- The dataset contains 180 patient records.
- The input dataset contains 14 columns, including the unique patient identifier.
- The target variable `heart_disease_present` is provided separately in the labels dataset.
- The values and labels datasets were merged using `patient_id`.
- No missing values were found in the provided dataset.
- No duplicate records were found.
- `patient_id` is a unique identifier and was excluded from Machine Learning features.
- `thal` is a categorical feature with three categories: normal, fixed_defect, and reversible_defect.
- Numerical and categorical variables were analyzed using distributions, boxplots, countplots, and correlation analysis.
- Outliers were identified using the IQR method. Since medical observations may represent genuine patient conditions, outliers were not automatically removed without domain justification.
- The target distribution contains both classes and was considered during train-test splitting and model evaluation.

In [ ]:
X = df_model.drop(
    "heart_disease_present",
    axis=1
)

y = df_model["heart_disease_present"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5
    )
}

In [ ]:
trained_models = {}
results = []

for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_probability = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_probability)
    })

    trained_models[model_name] = pipeline

print("All models trained successfully.")

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1 Score": "{:.4f}",
    "ROC-AUC": "{:.4f}"
})

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="Accuracy"
)

plt.title("Model Accuracy Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="Precision"
)

plt.title("Model Precision Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="Recall"
)

plt.title("Model Recall Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="F1 Score"
)

plt.title("Model F1 Score Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="ROC-AUC"
)

plt.title("Model ROC-AUC Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
for model_name, model in trained_models.items():

    y_pred = model.predict(X_test)

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred
    )

    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

In [ ]:
for model_name, model in trained_models.items():

    y_pred = model.predict(X_test)

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(9, 7))

for model_name, model in trained_models.items():

    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=model_name
    )

plt.title("ROC Curve Comparison")
plt.show()

In [ ]:
best_model_row = results_df.loc[
    results_df["F1 Score"].idxmax()
]

best_model_name = best_model_row["Model"]

best_model = trained_models[best_model_name]

print("Candidate Production Model:", best_model_name)

In [ ]:
print("Candidate Model:", best_model_name)
print("Accuracy:", round(best_model_row["Accuracy"], 4))
print("Precision:", round(best_model_row["Precision"], 4))
print("Recall:", round(best_model_row["Recall"], 4))
print("F1 Score:", round(best_model_row["F1 Score"], 4))
print("ROC-AUC:", round(best_model_row["ROC-AUC"], 4))

# Production Model Selection

Multiple Machine Learning classification algorithms were evaluated using Accuracy, Precision, Recall, F1 Score and ROC-AUC.

The candidate production model was selected based on the F1 Score as a balance between precision and recall. Recall and the confusion matrix were also considered because the project involves identifying potential heart disease cases.

The selected model should be considered a candidate for further validation rather than an independently deployable clinical diagnostic system. Before real-world healthcare use, the model would require validation using larger and representative patient populations, clinical review, monitoring, and appropriate regulatory and privacy controls.

In [ ]:
rf_pipeline = trained_models["Random Forest"]

rf_model = rf_pipeline.named_steps["model"]

feature_names = (
    rf_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
)

feature_importance.head(15)

In [ ]:
top_features = (
    feature_importance
    .head(10)
    .sort_values(by="Importance")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")

plt.show()

# Challenges Faced and Techniques Used

| Challenge | Technique Used | Reason |
|---|---|---|
| Separate feature and label datasets | Merged `values.csv` and `labels.csv` using `patient_id` | To create a complete modelling dataset |
| Patient identifier | Removed `patient_id` before modelling | It is a unique identifier and does not represent a useful medical feature |
| Categorical feature | One-Hot Encoding for `thal` | To convert categorical values into machine-learning-compatible numerical features |
| Feature scaling | StandardScaler | To place numerical features on comparable scales for scale-sensitive algorithms |
| Outlier identification | IQR and boxplot analysis | To identify unusual observations |
| Outlier treatment | Outliers were reviewed rather than automatically deleted | Extreme medical measurements may represent genuine patient observations |
| Model selection | Multiple classification algorithms were compared | To evaluate different approaches on the same dataset |
| Model evaluation | Accuracy, Precision, Recall, F1 Score, ROC-AUC and confusion matrix | To obtain a broader view of classification performance |
| Potential false negatives | Recall and confusion matrix analysis | Missing a potential positive case is important in a healthcare screening context |
| Small dataset | Careful train-test splitting and recognition of validation limitations | Model performance may vary when evaluated on a larger population |

# Suggestions to Hospitals

### 1. Early Risk Screening
Use the Machine Learning model as a screening and decision-support tool to identify patients who may require additional clinical assessment.

### 2. Monitor Relevant Patient Features
Hospitals can monitor important patient characteristics identified through exploratory data analysis and model feature analysis.

### 3. Risk-Based Alerts
Potentially high-risk predictions can be used to generate alerts for healthcare professionals for further review.

### 4. Improve Data Quality
Hospitals should maintain accurate, complete and consistent patient records because prediction performance depends on data quality.

### 5. Regular Model Evaluation
The model should be periodically evaluated using new and representative patient data to identify performance changes.

### 6. Human Clinical Review
Predictions should support healthcare professionals rather than replace clinical diagnosis or professional medical judgment.

### 7. Patient Privacy
Any real-world implementation should follow applicable healthcare data privacy, security and governance requirements.

# Limitations

- The dataset contains only 180 patient records, which is relatively small for developing a healthcare prediction system.
- The model performance may vary when applied to a different patient population.
- The dataset contains a limited set of clinical features.
- A train-test split provides only one evaluation sample; additional validation on larger datasets would be required before real-world deployment.
- The Machine Learning model should not be considered a standalone medical diagnostic system.

# Conclusion

The PRCP-1016 Heart Disease Prediction project analyzed patient-level clinical and exercise-related features to understand patterns associated with the presence of heart disease.

The feature dataset and label dataset were merged using `patient_id`. Data quality was examined through missing-value analysis, duplicate detection, descriptive statistics, visualization, correlation analysis and outlier analysis.

The patient identifier was removed before Machine Learning modelling. Categorical data was encoded and numerical features were scaled through a preprocessing pipeline.

The following classification algorithms were developed and compared:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine
5. K-Nearest Neighbors

The models were evaluated using Accuracy, Precision, Recall, F1 Score, ROC-AUC and confusion matrices. The candidate production model was selected based on the evaluation results, with particular consideration given to the balance between precision and recall.

The resulting system can be considered a potential decision-support and screening approach. Further validation using larger and representative clinical datasets would be required before considering real-world healthcare deployment.

In [ ]:
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Model comparison results saved successfully.")

In [ ]:
import joblib

joblib.dump(
    best_model,
    "heart_disease_prediction_model.pkl"
)

print("Candidate model saved successfully.")